# 01 Data Fusion and Exploration (Optimized)

Build a unified payload dataset from `data/raw` by standardizing all sources to two columns: `payload` and `label`.

**Optimizations:**
- Sample large files to prevent memory issues
- Progress tracking
- Memory management

In [1]:
import pandas as pd
from pathlib import Path
import gc

RAW_DIR = Path('../data/raw')

# Common column-name candidates across your raw files
PAYLOAD_CANDIDATES = ['Sentence', 'sentence', 'URL', 'url', 'payload', 'request']
LABEL_CANDIDATES = ['Label', 'label', 'classification', 'Class', 'class']

# For tabular CICIDS-style files with no payload text column:
# turn each row into a text payload from all non-label columns.
INCLUDE_TABULAR_AS_TEXT = True

# Memory optimization settings
MAX_ROWS_PER_FILE = 50000  # Limit rows per file to prevent memory issues
LARGE_FILE_THRESHOLD_MB = 50  # Files larger than this will be sampled

In [2]:
def read_csv_flexible(path: Path, **kwargs) -> pd.DataFrame:
    """Read CSV with flexible encoding detection and optional parameters."""
    for encoding in [None, 'utf-16', 'utf-8-sig', 'latin1']:
        try:
            if encoding is None:
                return pd.read_csv(path, **kwargs)
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except Exception:
            continue
    raise ValueError(f'Could not read {path}')


def find_label_column(df: pd.DataFrame):
    for candidate in LABEL_CANDIDATES:
        if candidate in df.columns:
            return candidate
    # fallback: match after stripping spaces (for columns like ' Label')
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for candidate in LABEL_CANDIDATES:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]
    return None


def find_payload_column(df: pd.DataFrame):
    for candidate in PAYLOAD_CANDIDATES:
        if candidate in df.columns:
            return candidate
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for candidate in PAYLOAD_CANDIDATES:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]
    return None


def row_to_payload_text_optimized(df: pd.DataFrame, label_col: str, max_features: int = 20) -> pd.Series:
    """Optimized version that limits the number of features to prevent extremely long payloads."""
    feature_cols = [c for c in df.columns if c != label_col]
    
    # Limit features to prevent memory explosion
    if len(feature_cols) > max_features:
        # Select most important features (non-constant columns with reasonable variance)
        numeric_cols = df[feature_cols].select_dtypes(include=['number']).columns
        if len(numeric_cols) > 0:
            # Use columns with highest variance
            variances = df[numeric_cols].var().sort_values(ascending=False)
            feature_cols = variances.head(max_features).index.tolist()
        else:
            feature_cols = feature_cols[:max_features]
    
    def serialize_row(row):
        parts = []
        for c in feature_cols:
            val = row[c]
            if pd.notna(val) and str(val).strip() != '':
                # Truncate very long values
                val_str = str(val)[:100]
                parts.append(f"{str(c).strip()}={val_str}")
        return " | ".join(parts)

    return df.apply(serialize_row, axis=1)


def load_and_standardize_optimized(raw_dir=RAW_DIR):
    """Optimized version with memory management and progress tracking."""
    combined_list = []
    report_rows = []
    
    # Get file info first
    files = list(raw_dir.glob('*.csv'))
    print(f"Found {len(files)} CSV files to process")
    
    for i, file_path in enumerate(sorted(files), 1):
        file_name = file_path.name
        file_size_mb = file_path.stat().st_size / (1024 * 1024)
        
        print(f"\n[{i}/{len(files)}] Processing {file_name} ({file_size_mb:.1f}MB)")

        try:
            # Handle large files by sampling
            if file_size_mb > LARGE_FILE_THRESHOLD_MB:
                print(f"  Large file detected - sampling first {MAX_ROWS_PER_FILE:,} rows")
                temp_df = read_csv_flexible(file_path, nrows=MAX_ROWS_PER_FILE)
            else:
                temp_df = read_csv_flexible(file_path)
                
        except Exception as ex:
            print(f"  ❌ Failed to read: {type(ex).__name__}")
            report_rows.append({
                'file': file_name,
                'status': 'skipped',
                'reason': f'read_error: {type(ex).__name__}',
                'rows_processed': 0,
                'file_size_mb': file_size_mb
            })
            continue

        print(f"  Loaded {len(temp_df):,} rows, {len(temp_df.columns)} columns")
        
        label_col = find_label_column(temp_df)
        if label_col is None:
            print(f"  ❌ No label column found")
            report_rows.append({
                'file': file_name,
                'status': 'skipped',
                'reason': 'no label column detected',
                'rows_processed': 0,
                'file_size_mb': file_size_mb
            })
            continue

        payload_col = find_payload_column(temp_df)

        if payload_col is not None:
            print(f"  ✓ Found payload column: {payload_col}")
            temp_out = temp_df[[payload_col, label_col]].rename(
                columns={payload_col: 'payload', label_col: 'label'}
            )
            source_mode = f'direct:{payload_col}'
        elif INCLUDE_TABULAR_AS_TEXT:
            print(f"  🔄 Converting tabular data to text (using optimized method)...")
            temp_out = pd.DataFrame({
                'payload': row_to_payload_text_optimized(temp_df, label_col),
                'label': temp_df[label_col]
            })
            source_mode = 'tabular_to_text_optimized'
        else:
            print(f"  ❌ No payload column and tabular conversion disabled")
            report_rows.append({
                'file': file_name,
                'status': 'skipped',
                'reason': 'no payload column and tabular conversion disabled',
                'rows_processed': 0,
                'file_size_mb': file_size_mb
            })
            continue

        # Clean up temp_df to free memory
        del temp_df
        gc.collect()
        
        temp_out['source_file'] = file_name
        combined_list.append(temp_out)
        
        report_rows.append({
            'file': file_name,
            'status': 'included',
            'reason': source_mode,
            'rows_processed': len(temp_out),
            'file_size_mb': file_size_mb
        })
        
        print(f"  ✓ Processed {len(temp_out):,} rows")

    if not combined_list:
        raise ValueError('No datasets could be loaded. Check raw files/columns.')

    print(f"\n🔄 Combining {len(combined_list)} datasets...")
    master = pd.concat(combined_list, ignore_index=True)
    
    print(f"🔄 Cleaning data (removing duplicates and empty values)...")
    initial_rows = len(master)
    master = master.drop_duplicates()
    master = master.dropna(subset=['payload', 'label']).copy()
    master['payload'] = master['payload'].astype(str).str.strip()
    master['label'] = master['label'].astype(str).str.strip()
    master = master[master['payload'] != '']
    
    final_rows = len(master)
    print(f"✓ Removed {initial_rows - final_rows:,} duplicate/empty rows")

    report_df = pd.DataFrame(report_rows)
    print(f"\n🎉 Final dataset shape: {master.shape}")
    return master.reset_index(drop=True), report_df

In [3]:
# Master Dataset + ingestion report (OPTIMIZED VERSION)
master_df, ingestion_report = load_and_standardize_optimized()

print("\n" + "="*50)
print("INGESTION REPORT")
print("="*50)
display(ingestion_report)

print("\n" + "="*50)
print("SAMPLE DATA")
print("="*50)
display(master_df.head())

Found 13 CSV files to process

[1/13] Processing csic_database.csv (28.2MB)
  Loaded 61,065 rows, 17 columns
  ✓ Found payload column: URL
  ✓ Processed 61,065 rows

[2/13] Processing Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv (73.6MB)
  Large file detected - sampling first 50,000 rows
  Loaded 50,000 rows, 79 columns
  🔄 Converting tabular data to text (using optimized method)...


d:\Documents\Projects\ModSecurity\venv\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


  ✓ Processed 50,000 rows

[3/13] Processing Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (73.3MB)
  Large file detected - sampling first 50,000 rows
  Loaded 50,000 rows, 79 columns
  🔄 Converting tabular data to text (using optimized method)...


d:\Documents\Projects\ModSecurity\venv\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


  ✓ Processed 50,000 rows

[4/13] Processing Friday-WorkingHours-Morning.pcap_ISCX.csv (55.6MB)
  Large file detected - sampling first 50,000 rows
  Loaded 50,000 rows, 79 columns
  🔄 Converting tabular data to text (using optimized method)...


...

d:\Documents\Projects\ModSecurity\venv\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


  ✓ Processed 50,000 rows

[13/13] Processing XSS_dataset.csv (1.6MB)
  Loaded 13,686 rows, 3 columns
  ✓ Found payload column: Sentence
  ✓ Processed 13,686 rows

🔄 Combining 13 datasets...
🔄 Cleaning data (removing duplicates and empty values)...
✓ Removed 146,296 duplicate/empty rows

🎉 Final dataset shape: (517701, 3)

INGESTION REPORT


,file,status,reason,rows_processed,file_size_mb
0,csic_database.csv,included,direct:URL,61065,28.171148
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,included,tabular_to_text_optimized,50000,73.551044
2,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,included,tabular_to_text_optimized,50000,73.343437
3,Friday-WorkingHours-Morning.pcap_ISCX.csv,included,tabular_to_text_optimized,50000,55.615163
4,Monday-WorkingHours.pcap_ISCX.csv,included,tabular_to_text_optimized,50000,168.731611
5,sqli.csv,included,direct:Sentence,4200,0.689651
6,sqliv2.csv,included,direct:Sentence,33761,3.442581
7,SQLiV3.csv,included,direct:Sentence,30919,2.210310
8,Thursday-WorkingHours-Afternoon-Infilteration....,included,tabular_to_text_optimized,50000,79.252659
9,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,included,tabular_to_text_optimized,170366,49.613250



SAMPLE DATA


,payload,label,source_file
0,http://localhost:8080/tienda1/index.jsp HTTP/1.1,0,csic_database.csv
1,http://localhost:8080/tienda1/publico/anadir.j...,0,csic_database.csv
2,http://localhost:8080/tienda1/publico/anadir.j...,0,csic_database.csv
3,http://localhost:8080/tienda1/publico/autentic...,0,csic_database.csv
4,http://localhost:8080/tienda1/publico/autentic...,0,csic_database.csv


In [4]:
print('Shape:', master_df.shape)
print('\nLabel distribution:')
display(master_df['label'].value_counts(dropna=False))

print('\nRows per source:')
display(master_df['source_file'].value_counts())

print('\nIngestion status count:')
display(ingestion_report['status'].value_counts())

print('\nTotal rows processed by file:')
display(ingestion_report[['file', 'rows_processed', 'file_size_mb']].sort_values('file_size_mb', ascending=False))

Shape: (517701, 3)

Label distribution:


label
BENIGN                                                                   392224
0                                                                         52994
1                                                                         39699
DDoS                                                                      23864
FTP-Patator                                                                4083
DoS slowloris                                                              2503
Web Attack � Brute Force                                                   1427
Web Attack � XSS                                                            652
PortScan                                                                    194
Web Attack � Sql Injection                                                   20
--                                                                           11
waitfor delay '0:0:__TIME__'--                                                4
SELECT * FROM Customers           


Rows per source:


source_file
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv         135608
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                46372
Monday-WorkingHours.pcap_ISCX.csv                               43824
Tuesday-WorkingHours.pcap_ISCX.csv                              43094
Wednesday-workingHours.pcap_ISCX.csv                            42593
Friday-WorkingHours-Morning.pcap_ISCX.csv                       41954
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv            35918
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv     35606
sqliv2.csv                                                      33727
SQLiV3.csv                                                      30635
csic_database.csv                                               13506
XSS_dataset.csv                                                 10914
sqli.csv                                                         3950
Name: count, dtype: int64


Ingestion status count:


status
included    13
Name: count, dtype: int64


Total rows processed by file:


,file,rows_processed,file_size_mb
11,Wednesday-workingHours.pcap_ISCX.csv,50000,214.735408
4,Monday-WorkingHours.pcap_ISCX.csv,50000,168.731611
10,Tuesday-WorkingHours.pcap_ISCX.csv,50000,128.821368
8,Thursday-WorkingHours-Afternoon-Infilteration....,50000,79.252659
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,50000,73.551044
2,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,50000,73.343437
3,Friday-WorkingHours-Morning.pcap_ISCX.csv,50000,55.615163
9,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,49.613250
0,csic_database.csv,61065,28.171148
6,sqliv2.csv,33761,3.442581


In [5]:
# Optional: save fused dataset
out_path = Path('../data/processed/master_fused_payloads.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
master_df.to_csv(out_path, index=False)
print(f'Saved: {out_path.resolve()}')
print(f'File size: {out_path.stat().st_size / (1024*1024):.1f}MB')

Saved: D:\Documents\Projects\ModSecurity\data\processed\master_fused_payloads.csv
File size: 200.1MB
